# LevelForge — GRPO Training Notebook

**Meta PyTorch OpenEnv Hackathon × Scaler School of Technology, India 2026**

This notebook trains a 0.5B language model (Qwen2.5) to design personality-aware 2D platformer levels using GRPO reinforcement learning. The reward signal comes from A* pathfinding math — no LLM judge needed.

---

### Three training runs covered in this notebook:
| Run | Steps | Purpose |
|-----|-------|--------|
| **Run 1** | 200 steps | Baseline — observe initial learning |
| **Run 2** | 500 steps | Extended — model converges, reward peaks at 1.997 |
| **Run 3** | 200 steps | Curriculum — self-improving adaptive difficulty |

### Links
- 🤗 **HF Space:** https://huggingface.co/spaces/pranavgadodia/levelforge-env
- 💻 **GitHub:** https://github.com/Pranavpro23/levelforge_env

---

> **Hardware requirement:** GPU runtime required (T4 or better). Go to Runtime → Change runtime type → T4 GPU

## Cell 1 — Install Dependencies

Install all required packages. Run once per session.

- `unsloth` — fast QLoRA fine-tuning (2x speedup over standard HF)
- `trl` — GRPO trainer from HuggingFace
- `openenv-core` — OpenEnv framework
- `requests` — HTTP calls to LevelForge HF Space
- `matplotlib`, `Pillow` — plotting reward curves and images

In [ ]:
# Verify GPU is available before installing
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    print(result.stdout.split('\n')[8])  # GPU name line
else:
    print('❌ No GPU found — go to Runtime → Change runtime type → T4 GPU')

!pip install -q unsloth trl openenv-core requests matplotlib Pillow datasets
print('✅ All packages installed')

## Cell 2 — Configuration

Set all training parameters here. Change `MAX_STEPS` to switch between runs:
- `MAX_STEPS = 200` → Run 1 (baseline) or Run 3 (curriculum)
- `MAX_STEPS = 500` → Run 2 (extended)

**Important:** `ENV_URL` must point to the API endpoint, not the HF Space webpage.

In [ ]:
# ─── Training Configuration ───────────────────────────────────────────────

# Model: Qwen2.5-0.5B with Unsloth QLoRA acceleration
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"

# HF Space API URL — must use .hf.space domain NOT huggingface.co/spaces/...
ENV_URL = "https://pranavgadodia-levelforge-env.hf.space"

# Training steps: 200 for baseline/curriculum, 500 for extended run
MAX_STEPS = 500

# Output directory for checkpoints (saved every 50 steps)
OUTPUT_DIR = "levelforge_out"

# ─── Verify ENV_URL is reachable ──────────────────────────────────────────
import requests
try:
    r = requests.get(f"{ENV_URL}/health", timeout=15)
    if r.status_code == 200:
        print(f'✅ ENV_URL reachable: {ENV_URL}')
        print(f'   Health: {r.json()}')
    else:
        print(f'⚠️  ENV_URL returned {r.status_code} — check if Space is running')
except Exception as e:
    print(f'❌ Cannot reach ENV_URL: {e}')
    print('   Make sure the HF Space is running at https://huggingface.co/spaces/pranavgadodia/levelforge-env')

print(f'\nConfiguration:')
print(f'  Model:     {MODEL_NAME}')
print(f'  MAX_STEPS: {MAX_STEPS}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')

## Cell 3 — Load Model

Load Qwen2.5-0.5B-Instruct with:
- **4-bit quantization** — fits in 15GB T4 VRAM
- **QLoRA** — only trains 3.44% of parameters (17M of 511M)
- **Unsloth** — 2x faster training than standard HuggingFace

Expected output: Model loads with ~22GB memory on A10G, ~15GB on T4.

In [ ]:
import torch
from unsloth import FastLanguageModel

print('Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,         # 4-bit quantization for memory efficiency
    fast_inference=False,
    max_lora_rank=32,
    gpu_memory_utilization=0.6
)

# Apply LoRA adapters — only these layers are trained
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                      # LoRA rank
    lora_alpha=32,             # LoRA alpha (same as rank = balanced)
    target_modules=[           # Which layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing="unsloth",  # Memory-efficient backprop
    random_state=3407
)

print(f'\n✅ Model loaded successfully')
print(f'   Parameters: {model.num_parameters():,} total')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'   Trainable:  {trainable:,} ({100*trainable/model.num_parameters():.2f}%)')
print(f'   GPU memory: {torch.cuda.memory_allocated()/1e9:.1f}GB used')

## Cell 4 — System Prompt

The system prompt instructs the model how to design levels. Key requirements:
1. **Think first** — wrap reasoning in `<think>...</think>` tags (this is rewarded)
2. **JSON output** — edits must be valid JSON format
3. **Personality awareness** — design differently for brave/cautious/explorer

The format_reward function checks for `<think>` tags — this is the first signal the model learns.

In [ ]:
SYSTEM_PROMPT = """You are a game level designer for a 2D platformer.
You are given an 8x16 grid and must place tiles to create a fun level.

Tiles:
  . = empty space
  # = wall or platform
  ^ = spike (kills player, blocks A* path)
  $ = coin (collectible)
  E = enemy (hazard)
  P = player start (FIXED — do not move)
  G = goal (FIXED — do not move)

PERSONALITY GUIDE:
  brave    → place many spikes (^) and enemies (E), narrow dangerous paths
  cautious → wide open paths, few hazards, comfortable navigation
  explorer → multiple branching routes, hidden coins at different heights

OUTPUT FORMAT — follow exactly:

<think>
Which personality am I designing for? What makes a good level for them?
</think>
{"edits": [{"row": N, "col": N, "tile": "X"}, ...], "declare_done": false}

Rules:
- Always use <think>...</think> before your JSON
- Place 3-6 tiles per step
- Do NOT place tiles on P or G positions
- declare_done: true only on your final step"""

print('✅ System prompt defined')
print(f'   Length: {len(SYSTEM_PROMPT)} characters')
print(f'   Preview: {SYSTEM_PROMPT[:100]}...')

## Cell 5 — Capture Baseline Outputs (BEFORE Training)

Run this BEFORE any training to capture what the untrained model produces.
This is your "before" evidence for the before/after comparison.

**Expected baseline behaviour:**
- Generic language same for all 3 personalities
- No `<think>` tags (format not learned yet)
- Random or no tile placements
- No personality-specific reasoning

In [ ]:
import torch

baseline_prompts = [
    "Design a first_steps level for a brave player. Target path length: 16.",
    "Design a first_steps level for a cautious player. Target path length: 16.",
    "Design a first_steps level for an explorer player. Target path length: 16.",
]

baseline_outputs = []
print('Capturing baseline outputs (untrained model)...')
print('=' * 60)

for p in baseline_prompts:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": p}],
        tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    baseline_outputs.append(text)

    print(f'\n>>> BASELINE — {p[:45]}')
    print(text[:300])

print('\n' + '=' * 60)
print('✅ Baseline captured — compare with trained outputs after training')
print('   Key things to look for after training:')
print('   - Does the model use <think> tags?')
print('   - Does each personality get different reasoning?')
print('   - Is the JSON output more structured?')

## Cell 6 — Reward Functions

Two reward functions used by GRPOTrainer:

**`format_reward`** — checks for `<think>` XML tags (dense signal)
- Returns ~0.998 if both `<think>` and `</think>` present
- Returns ~0.499 if only one tag present
- Returns 1e-6 if no tags (epsilon-clamped — never exact 0)
- Model learns this first (steps 0-50)

**`call_env_reward`** — calls the live HF Space /step endpoint
- Extracts task and personality from prompt text
- Resets environment, takes a step, returns reward.total
- Includes solvability (A*), difficulty, personality match, etc.
- Model learns this after format is established (steps 50+)

In [ ]:
import requests, json

# Persistent session keeps connection alive between reward calls
session = requests.Session()

def format_reward(prompts, completions, **kwargs):
    """Reward model for using <think> XML reasoning tags.
    Dense signal — model learns this first in training.
    Epsilon-clamped to avoid exact 0/1 which fails OpenEnv validator."""
    scores = []
    for c in completions:
        text = c[0]["content"]
        has_open  = "<think>"  in text
        has_close = "</think>" in text
        raw = (0.499 if has_open else 0) + (0.499 if has_close else 0)
        scores.append(max(1e-6, min(1.0 - 1e-6, raw)))
    return scores


def call_env_reward(prompts, completions, **kwargs):
    """Reward from live LevelForge HF Space environment.
    Calls /reset then /step for each completion.
    Returns reward.total which combines A*, difficulty, personality, etc."""
    rewards = []

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        try:
            # Extract task and personality from the user message text
            user_msg = ""
            for msg in prompt:
                if msg["role"] == "user":
                    user_msg = msg["content"]
                    break

            task        = "first_steps"
            personality = "brave"
            if "gap_jumper"       in user_msg: task = "gap_jumper"
            elif "symmetric_shrine" in user_msg: task = "symmetric_shrine"
            if "cautious" in user_msg: personality = "cautious"
            elif "explorer" in user_msg: personality = "explorer"

            # Wake up Space + reset environment
            session.get(f"{ENV_URL}/health", timeout=20)
            session.post(f"{ENV_URL}/reset",
                json={"task_name": task, "personality": personality},
                timeout=20)

            # Send model completion as a step
            text = completion[0]["content"]
            resp = session.post(f"{ENV_URL}/step",
                json={"reasoning": text[:400], "edits": [], "declare_done": False},
                timeout=20)

            data  = resp.json()
            total = float(data["reward"]["total"])
            rewards.append(max(1e-6, min(1.0 - 1e-6, total)))

        except Exception as e:
            print(f"env_reward error {i}: {e}")
            rewards.append(1e-6)

    return rewards


# ─── Quick sanity test ────────────────────────────────────────────────────
print('Testing reward functions...')

test_prompt = [[{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": "Design a first_steps level for a brave player."}]]
test_completion = [[{"role": "assistant",
                     "content": "<think>Placing spikes for brave</think>\n{\"edits\":[{\"row\":5,\"col\":3,\"tile\":\"^\"}],\"declare_done\":false}"}]]

fmt = format_reward(test_prompt, test_completion)
print(f'  format_reward: {fmt[0]:.4f} (expected ~0.998)')

env = call_env_reward(test_prompt, test_completion)
print(f'  env_reward:    {env[0]:.4f} (expected ~0.85-0.95)')

assert env[0] > 0.001, '❌ env_reward is 1e-6 — check ENV_URL and Space health'
print('✅ Both reward functions working correctly')

## Cell 7 — Build Training Dataset

Creates 200 training prompts across 3 tasks × 3 personalities.

Each prompt includes:
- Task name (first_steps / gap_jumper / symmetric_shrine)
- Personality (brave / cautious / explorer)
- Target path length
- Personality-specific design hints

9 unique scenarios × 23 repeats = 200 total prompts.

In [ ]:
from datasets import Dataset

SCENARIOS = [
    # first_steps — easy: pre-built floor, add coins + hazards
    {"task_name": "first_steps", "personality": "brave",    "target_path_len": 16, "target_coin_count": 3},
    {"task_name": "first_steps", "personality": "cautious", "target_path_len": 16, "target_coin_count": 3},
    {"task_name": "first_steps", "personality": "explorer", "target_path_len": 16, "target_coin_count": 3},
    # gap_jumper — medium: gaps in floor, add platforms + enemy
    {"task_name": "gap_jumper",  "personality": "brave",    "target_path_len": 22, "target_coin_count": 4},
    {"task_name": "gap_jumper",  "personality": "cautious", "target_path_len": 22, "target_coin_count": 4},
    {"task_name": "gap_jumper",  "personality": "explorer", "target_path_len": 22, "target_coin_count": 4},
    # symmetric_shrine — hard: design complete level from scratch
    {"task_name": "symmetric_shrine", "personality": "brave",    "target_path_len": 30, "target_coin_count": 6},
    {"task_name": "symmetric_shrine", "personality": "cautious", "target_path_len": 30, "target_coin_count": 6},
    {"task_name": "symmetric_shrine", "personality": "explorer", "target_path_len": 30, "target_coin_count": 6},
]

PERSONALITY_HINTS = {
    "brave":    "Place many spikes (^) and enemies (E) for a dangerous, exciting level. Narrow paths are good.",
    "cautious": "Keep paths wide and open. Use few spikes. Make it safe and comfortable to navigate.",
    "explorer": "Create multiple branching paths with hidden coins ($) in unexpected places.",
}

data = []
for s in (SCENARIOS * 23)[:200]:
    hint     = PERSONALITY_HINTS[s["personality"]]
    user_msg = (
        f"Task: {s['task_name']} | Personality: {s['personality']} | "
        f"Target path length: {s['target_path_len']} | Target coins: {s['target_coin_count']}\n"
        f"Player hint: {hint}\n"
        f"Design the level now. Use JSON edits format."
    )
    data.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        "task_name":   s["task_name"],
        "personality": s["personality"],
    })

dataset = Dataset.from_list(data)
print(f'✅ Dataset size: {len(dataset)}')
print(f'✅ Tasks: {set(d["task_name"] for d in data)}')
print(f'✅ Personalities: {set(d["personality"] for d in data)}')
print(f'✅ Sample prompt:\n   {data[0]["prompt"][1]["content"][:120]}')

## Cell 8 — GRPO Training

**Run 1 (200 steps):** Set `MAX_STEPS = 200` in Cell 2. Baseline run to observe initial learning.

**Run 2 (500 steps):** Set `MAX_STEPS = 500` in Cell 2. Extended run — model converges. format_reward hits 0.998 by step 50, total reward peaks at 1.997.

**Expected learning phases:**
- Steps 0–50: format_reward rises (model learns `<think>` tags)
- Steps 50–200: env_reward rises (model learns solvable level design)
- Steps 200–500: both rewards stable and high (model converges)

Checkpoints saved every 50 steps to `levelforge_out/`. If session disconnects, resume from the latest checkpoint.

> ⏱️ **Estimated time:** ~105 min for 200 steps on T4, ~260 min for 500 steps on T4

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import os

# Check for existing checkpoint to resume from
resume_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = sorted([d for d in os.listdir(OUTPUT_DIR) if 'checkpoint' in d])
    if checkpoints:
        resume_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f'📂 Found checkpoint: {resume_checkpoint}')
        print('   Will resume training from this checkpoint')
    else:
        print('📂 No checkpoint found — starting fresh')
else:
    print('📂 No output dir — starting fresh')

print(f'\n🚀 Starting GRPO training: {MAX_STEPS} steps')
print(f'   Output dir: {OUTPUT_DIR}')
print(f'   Reward functions: format_reward + call_env_reward')
print('─' * 50)

args = GRPOConfig(
    # Optimizer
    learning_rate=5e-6,
    optim="paged_adamw_8bit",        # Memory-efficient optimizer

    # Batch configuration
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,               # GRPO samples 4 completions per step

    # Sequence lengths
    max_prompt_length=256,
    max_completion_length=768,

    # Training schedule
    max_steps=MAX_STEPS,
    save_steps=50,                   # Checkpoint every 50 steps
    logging_steps=1,                 # Log every step

    # GRPO specific
    beta=0.04,                       # KL penalty coefficient
    loss_type="dr_grpo",             # Dr-GRPO variant (more stable)
    mask_truncated_completions=True, # Don't penalise truncated outputs

    # Output
    report_to="none",                # Disable wandb/tensorboard
    output_dir=OUTPUT_DIR,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, call_env_reward],  # format first, env second
    args=args,
    train_dataset=dataset,
)

# Start or resume training
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

print('\n' + '=' * 50)
print('✅ Training complete!')
print(f'   Global steps:  {train_result.global_step}')
print(f'   Training loss: {train_result.training_loss:.6f}')
print(f'   Runtime:       {train_result.metrics["train_runtime"]:.0f}s ({train_result.metrics["train_runtime"]/3600:.1f}h)')

## Cell 9 — Save Model Checkpoint

Save the trained model after each run before starting the next one.
This preserves your progress and lets you compare runs.

In [ ]:
# Save checkpoint — rename based on which run just completed
checkpoint_name = f"levelforge_checkpoint_{MAX_STEPS}steps"

model.save_pretrained(checkpoint_name)
tokenizer.save_pretrained(checkpoint_name)

print(f'✅ Model saved to {checkpoint_name}/')
print(f'   Files: {[f for f in __import__("os").listdir(checkpoint_name)[:5]]}')
print(f'\nNext steps:')
print(f'  - Run Cell 10 to plot reward curves')
print(f'  - Run Cell 11 to capture trained model outputs')
print(f'  - To continue to next run: change MAX_STEPS in Cell 2 and re-run Cell 8')

## Cell 10 — Plot Reward Curves

Generate and save reward curve plots from the completed training run.

Two plots saved:
1. **`reward_curve.png`** — total reward over training steps
2. **`reward_curve_components.png`** — format_reward vs env_reward vs total

Both plots are committed to GitHub and embedded in the README.

In [ ]:
import matplotlib.pyplot as plt
import json

log_history = trainer.state.log_history
print(f'Total log entries: {len(log_history)}')

# Extract reward data
steps, total_r, fmt_r, env_r = [], [], [], []
for log in log_history:
    if 'reward' in log:
        steps.append(log.get('step', 0))
        total_r.append(log.get('reward', 0))
        fmt_r.append(log.get('rewards/format_reward/mean', 0))
        env_r.append(log.get('rewards/call_env_reward/mean', 0))

print(f'Steps with reward data: {len(steps)}')

if not steps:
    print('❌ No reward data found in log_history')
    print('   Keys available:', list(log_history[0].keys()) if log_history else 'empty')
else:
    # ─── Plot 1: Total reward curve ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(steps, total_r, 'b-o', linewidth=1.5, markersize=3, label='reward/mean')
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Reward (0-2 scale)')
    axes[0].set_title(f'LevelForge: Total Reward During GRPO Training ({MAX_STEPS} steps)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(steps, fmt_r, 'g-o', linewidth=1.5, markersize=3, label='format_reward (XML tags)')
    axes[1].plot(steps, env_r, 'r-o', linewidth=1.5, markersize=3, label='env_reward (A* solvable)')
    axes[1].set_xlabel('Training Step')
    axes[1].set_ylabel('Reward (0-1 scale)')
    axes[1].set_title('LevelForge: Component Rewards')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved reward_curve.png')

    # ─── Plot 2: Component progression (better storytelling plot) ────────
    fig2, ax2 = plt.subplots(figsize=(14, 6))
    ax2.plot(steps, fmt_r,  'g-o', linewidth=1.5, markersize=2, label='Format reward (XML tags)')
    ax2.plot(steps, env_r,  'r-o', linewidth=1.5, markersize=2, label='Env reward (A* solvable)')
    ax2.plot(steps, total_r,'b-',  linewidth=2.5,               label='Total reward')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Reward (0-1 scale)')
    ax2.set_title(f'LevelForge: Format → Env reward progression (GRPO, {MAX_STEPS} steps)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('reward_curve_components.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved reward_curve_components.png')

    # ─── Summary stats ───────────────────────────────────────────────────
    print(f'\nTraining Summary:')
    print(f'  Peak total reward:   {max(total_r):.4f} at step {steps[total_r.index(max(total_r))]}')
    print(f'  Final total reward:  {total_r[-1]:.4f}')
    print(f'  Final format_reward: {fmt_r[-1]:.4f}')
    print(f'  Final env_reward:    {env_r[-1]:.4f}')

    # ─── Save full training log ───────────────────────────────────────────
    with open('training_logs.json', 'w') as f:
        json.dump(log_history, f, indent=2)
    print(f'\n✅ Saved training_logs.json ({len(log_history)} log entries)')

## Cell 11 — Capture Trained Model Outputs (AFTER Training)

Run the same 3 prompts as Cell 5 and compare outputs.

**What to look for after training:**
- ✅ `<think>` tags present (format learned)
- ✅ Different reasoning for each personality
- ✅ More structured JSON output
- ✅ Brave: mentions spikes/enemies/danger
- ✅ Cautious: mentions safe/open/comfortable
- ✅ Explorer: mentions branching/hidden/exploration

In [ ]:
import torch

trained_outputs = []
print('Capturing trained model outputs...')
print('=' * 60)

for p in baseline_prompts:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": p}],
        tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=300, temperature=0.7, do_sample=True)

    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    trained_outputs.append(text)

    print(f'\n>>> TRAINED — {p[:45]}')
    print(text[:350])

print('\n' + '=' * 60)
print('BEFORE vs AFTER COMPARISON')
print('=' * 60)

for i, (b, t, p) in enumerate(zip(baseline_outputs, trained_outputs, baseline_prompts)):
    personality = p.split('for a ')[1].split(' player')[0]
    print(f'\n[{personality.upper()}]')
    print(f'BEFORE: {b[:180]}')
    print(f'AFTER:  {t[:180]}')

print('\n✅ Comparison complete — screenshot this output for your README evidence')

## Cell 12 — Curriculum Training (Run 3)

After completing Run 2 (500 steps), run this curriculum training.

**What curriculum training does:**
- Starts at tutorial difficulty (avg reward < 0.3)
- Automatically escalates: tutorial → easy → medium → hard → tricky
- Uses rolling reward history to decide current difficulty
- Directly demonstrates Theme 4: Self-Improvement

**Expected behaviour:**
- Model enters already competent (from Run 2)
- Starts at reward ~1.4 from step 1 (no warm-up needed)
- format_reward hits 0.998 immediately
- env_reward sustains ~0.894

> ⚠️ **Do not run this before Run 2** — curriculum training benefits from the 500-step trained weights.

In [ ]:
from datasets import Dataset

# Curriculum dataset — all prompts use task_name='curriculum'
# The environment automatically selects difficulty based on rolling reward history
curriculum_data = []
for _ in range(200):
    curriculum_data.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content":
             "Design a level using curriculum mode. "
             "The environment will automatically increase difficulty as you improve. "
             "Start simple and adapt to the challenge."}
        ],
        "task_name":   "curriculum",
        "personality": "brave",
    })

curriculum_dataset = Dataset.from_list(curriculum_data)
print(f'✅ Curriculum dataset: {len(curriculum_dataset)} prompts')
print(f'   All use task_name=curriculum — difficulty auto-adapts')
print(f'\nCurriculum levels (defined in server/scenarios.py):')
print(f'  < 0.3  → tutorial    (straight path, 1 coin, no hazards)')
print(f'  0.3-0.5 → easy       (coins + 1-2 spikes)')
print(f'  0.5-0.65 → medium    (gaps, platforms, 1 enemy)')
print(f'  0.65-0.8 → hard      (design from scratch, symmetric)')
print(f'  0.8-0.9 → speedrunner/dungeon (random choice)')
print(f'  >= 0.9  → tricky     (looks easy, hidden complexity)')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

CURRICULUM_STEPS = 200

print(f'🚀 Starting Curriculum GRPO training: {CURRICULUM_STEPS} steps')
print('─' * 50)

curriculum_args = GRPOConfig(
    learning_rate=5e-6,
    optim="paged_adamw_8bit",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=768,
    max_steps=CURRICULUM_STEPS,
    save_steps=50,
    logging_steps=1,
    beta=0.04,
    loss_type="dr_grpo",
    mask_truncated_completions=True,
    report_to="none",
    output_dir="levelforge_curriculum_out",  # Separate dir from main runs
)

curriculum_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, call_env_reward],
    args=curriculum_args,
    train_dataset=curriculum_dataset,
)

curriculum_result = curriculum_trainer.train()

print('\n' + '=' * 50)
print('✅ Curriculum training complete!')
print(f'   Steps:  {curriculum_result.global_step}')
print(f'   Loss:   {curriculum_result.training_loss:.6f}')
print(f'   Time:   {curriculum_result.metrics["train_runtime"]/60:.0f} minutes')

## Cell 13 — Download Results & Commit to GitHub

Download all output files from the notebook environment and commit to GitHub.

Files to download:
- `reward_curve.png` — total reward plot
- `reward_curve_components.png` — format/env/total component plot
- `training_logs.json` — full step-by-step metrics

After downloading, run the git commands in your local terminal.

In [ ]:
import os

# List all output files
output_files = [
    'reward_curve.png',
    'reward_curve_components.png',
    'training_logs.json',
]

print('Output files generated:')
for f in output_files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status} {f} ({size/1024:.1f} KB)')

# Download from JupyterLab: right-click each file → Download
# OR from Colab:
try:
    from google.colab import files
    print('\nDownloading files (Colab)...')
    for f in output_files:
        if os.path.exists(f):
            files.download(f)
            print(f'  Downloaded: {f}')
except ImportError:
    print('\nNot running in Colab — download files manually from JupyterLab file browser')
    print('Right-click each file → Download')

print('\n─── After downloading, run in your local terminal: ──────────────')
print('''
cd ~/levelforge_env
cp ~/Downloads/reward_curve.png .
cp ~/Downloads/reward_curve_components.png .
cp ~/Downloads/training_logs.json .
git add reward_curve.png reward_curve_components.png training_logs.json
git commit -m "add: training results - reward curves and logs"
git push origin main
''')

## Appendix A — Resume From Checkpoint

If your session disconnects mid-training, use this cell to resume from the last saved checkpoint.
Checkpoints are saved every 50 steps to `levelforge_out/`.

In [ ]:
import os

# Find available checkpoints
if os.path.exists(OUTPUT_DIR):
    checkpoints = sorted([d for d in os.listdir(OUTPUT_DIR) if 'checkpoint' in d])
    print(f'Available checkpoints in {OUTPUT_DIR}/')
    for cp in checkpoints:
        print(f'  {cp}')

    if checkpoints:
        latest = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f'\nLatest: {latest}')
        print('Run Cell 8 — it will automatically resume from this checkpoint')
    else:
        print('No checkpoints found — will start from scratch')
else:
    print(f'Output dir {OUTPUT_DIR} does not exist — will start from scratch')

# Note: Cell 8 already handles auto-resume — just re-run it after reconnecting

## Appendix B — Test Environment Directly

Use this cell to test the LevelForge HF Space API independently of training.
Useful for debugging reward function issues.

In [ ]:
import requests, json

print(f'Testing LevelForge environment at {ENV_URL}\n')

# 1. Health check
h = requests.get(f'{ENV_URL}/health', timeout=15)
print(f'Health:  {h.status_code} → {h.json()}')

# 2. Reset
r = requests.post(f'{ENV_URL}/reset',
    json={'task_name': 'first_steps', 'personality': 'brave'}, timeout=15)
obs = r.json()
print(f'Reset:   {r.status_code} → episode_id={obs["episode_id"][:8]}...')
print(f'         Grid row 6: {obs["grid"][6]}')

# 3. Step with a test action
s = requests.post(f'{ENV_URL}/step',
    json={'reasoning': '<think>Placing spike for brave player</think>',
          'edits': [{'row': 5, 'col': 4, 'tile': '^'}],
          'declare_done': False}, timeout=15)
step_data = s.json()
print(f'Step:    {s.status_code}')
print(f'Reward breakdown:')
for k, v in step_data['reward'].items():
    bar = '█' * int(float(v) * 20) if isinstance(v, (int, float)) else ''
    print(f'  {k:20s}: {float(v):.4f} {bar}')

# 4. Curriculum level
cl = requests.get(f'{ENV_URL}/curriculum_level', timeout=15)
print(f'\nCurriculum level: {cl.json()["curriculum_level"]["name"]} (avg_reward={cl.json()["avg_reward"]:.3f})')

print('\n✅ Environment fully functional')